Object-oriented programming (OOP) also comes in handy when you need to prepare your data for deep learning. OOP allows you to create custom data classes that integrates well with PyTorch's data processing. This can massively speed up your model training.

Your custom dataset class neeeds to have three components: The ``__init__``, ``__len__``, and ``__getitem__`` methods.

- In the `__init__` method, the dataset is loaded and converted to torch tensors. In other words, it's initialized.
- The `__len__` method returns the number of samples in your dataset.
- The `__getitem__` returns the data on a given index.

## Setup

### Import Libraries

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.utils.data import random_split

import pandas as pd

## Section 1: The ``__init__`` function

In this notebook, you'll learn how to create each of this methods and make a custom data class. You'll start with what goes into the ``__init__`` function.

| Code | Description |
|---|---|
| `df = pd.read_csv('data.csv')` | Reads the data in `data.csv` into a pandas DataFrame and assigns it to the variable `df`. |
| `df.values` | Get the values (and not the keys) from the DataFrame `df`. |
| `df['column_name'].values` | Get the values (and not the keys) from the column named `column_name` in the DataFrame `df`. |
| `df.drop(columns = 'column_name')` | Drops the data in the column named `column_name` from the DataFrame `df`. |
| `data = torch.tensor(data)` | Make `data` a PyTorch tensor. |
| `features = torch.tensor(features, dtype=torch.float32)` | Make `features` a PyTorch tensor of data type `float32`. |
| `labels = torch.tensor(labels, dtype=torch.long)` | Make `features` a PyTorch tensor of data type `long` (`long` is PyTorch's alias for `int64`). |

#### **Exercises**


**Example**: Load the dataset `titanic.csv` in the folder `data/raw` into a pandas DataFrame.

In [ ]:
df_titanic = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv')
df_titanic.head()

,survived,pclass,sex,age,fare,embark_town,deck
0,0,3,male,22.0,7.2500,Southampton,NaN
1,1,1,female,38.0,71.2833,Cherbourg,C
2,1,3,female,26.0,7.9250,Southampton,NaN
3,1,1,female,35.0,53.1000,Southampton,C
4,0,3,male,35.0,8.0500,Southampton,NaN
...,...,...,...,...,...,...,...
886,0,2,male,27.0,13.0000,Southampton,NaN
887,1,1,female,19.0,30.0000,Southampton,B
888,0,3,female,NaN,23.4500,Southampton,NaN
889,1,1,male,26.0,30.0000,Cherbourg,C


**Exercise**: Read the dataset `wine.csv` in the folder `data/raw` into a pandas DataFrame named `df`. Display `df`.

In [ ]:
df = pd.read_csv('data/raw/wine.csv')
df

,Wine,Alcohol,Malic.acid,Ash,Acl,Mg,Phenols,Flavanoids,Nonflavanoid.phenols,Proanth,Color.int,Hue,OD,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,3,13.71,5.65,2.45,20.5,95,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740
174,3,13.40,3.91,2.48,23.0,102,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750
175,3,13.27,4.28,2.26,20.0,120,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835
176,3,13.17,2.59,2.37,20.0,120,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840


**Exercise**: Get the data of the data in the column named `Wine` in the DataFrame and assign it to a variable named `labels`. Display `labels` to check that it contains the right data from the DataFrame.

In [ ]:
labels = df['Wine']
labels

0      1
1      1
2      1
3      1
4      1
      ..
173    3
174    3
175    3
176    3
177    3
Name: Wine, Length: 178, dtype: int64

**Exercise**: Get the data of all other columns in the DataFrame and assign them to a variable `features`. Display `features`.

**Hint**: The function `df.drop` is useful here.

In [ ]:
features = df.drop(columns='Wine')
features

,Alcohol,Malic.acid,Ash,Acl,Mg,Phenols,Flavanoids,Nonflavanoid.phenols,Proanth,Color.int,Hue,OD,Proline
0,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735
...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740
174,13.40,3.91,2.48,23.0,102,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750
175,13.27,4.28,2.26,20.0,120,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835
176,13.17,2.59,2.37,20.0,120,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840


**Exercise**: Get the `values` of `labels`.

In [ ]:
labels.values

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3])

**Exercise**: Get the `values` of `features`.

In [ ]:
features.values

array([[1.423e+01, 1.710e+00, 2.430e+00, ..., 1.040e+00, 3.920e+00,
        1.065e+03],
       [1.320e+01, 1.780e+00, 2.140e+00, ..., 1.050e+00, 3.400e+00,
        1.050e+03],
       [1.316e+01, 2.360e+00, 2.670e+00, ..., 1.030e+00, 3.170e+00,
        1.185e+03],
       ...,
       [1.327e+01, 4.280e+00, 2.260e+00, ..., 5.900e-01, 1.560e+00,
        8.350e+02],
       [1.317e+01, 2.590e+00, 2.370e+00, ..., 6.000e-01, 1.620e+00,
        8.400e+02],
       [1.413e+01, 4.100e+00, 2.740e+00, ..., 6.100e-01, 1.600e+00,
        5.600e+02]], shape=(178, 13))

**Example**: Convert the labels into a torch tensor.

**Hint**: Get the values of the `labels` when you make it into a tensor.

In [ ]:
labels_torch = torch.tensor(labels.values, dtype=torch.long)
labels_torch

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

**Exercise**: Convert the `features` into a torch tensor with data type `float64`.

In [ ]:
features_torch = torch.tensor(features.values, dtype=torch.float64)
features_torch

tensor([[1.4230e+01, 1.7100e+00, 2.4300e+00,  ..., 1.0400e+00, 3.9200e+00,
         1.0650e+03],
        [1.3200e+01, 1.7800e+00, 2.1400e+00,  ..., 1.0500e+00, 3.4000e+00,
         1.0500e+03],
        [1.3160e+01, 2.3600e+00, 2.6700e+00,  ..., 1.0300e+00, 3.1700e+00,
         1.1850e+03],
        ...,
        [1.3270e+01, 4.2800e+00, 2.2600e+00,  ..., 5.9000e-01, 1.5600e+00,
         8.3500e+02],
        [1.3170e+01, 2.5900e+00, 2.3700e+00,  ..., 6.0000e-01, 1.6200e+00,
         8.4000e+02],
        [1.4130e+01, 4.1000e+00, 2.7400e+00,  ..., 6.1000e-01, 1.6000e+00,
         5.6000e+02]], dtype=torch.float64)

**Exercise**: Convert the `features` into a torch tensor with data type `float32`.

In [ ]:
features_torch = torch.tensor(features.values, dtype=torch.float64)
features_torch

tensor([[1.4230e+01, 1.7100e+00, 2.4300e+00,  ..., 1.0400e+00, 3.9200e+00,
         1.0650e+03],
        [1.3200e+01, 1.7800e+00, 2.1400e+00,  ..., 1.0500e+00, 3.4000e+00,
         1.0500e+03],
        [1.3160e+01, 2.3600e+00, 2.6700e+00,  ..., 1.0300e+00, 3.1700e+00,
         1.1850e+03],
        ...,
        [1.3270e+01, 4.2800e+00, 2.2600e+00,  ..., 5.9000e-01, 1.5600e+00,
         8.3500e+02],
        [1.3170e+01, 2.5900e+00, 2.3700e+00,  ..., 6.0000e-01, 1.6200e+00,
         8.4000e+02],
        [1.4130e+01, 4.1000e+00, 2.7400e+00,  ..., 6.1000e-01, 1.6000e+00,
         5.6000e+02]], dtype=torch.float64)

**Exercise**: Put the code below into a function named `init` that takes the filepath as input and returns `features` and `labels`. Run the cell code calling the init function after you are done.

In [ ]:
# load data
df = pd.read_csv('https://gist.githubusercontent.com/tijptjik/9408623/raw/b237fa5848349a14a14e5d4107dc7897c21951f5/wine.csv')

features = df.drop(columns = 'Wine')
labels = df['Wine']

features = torch.tensor(features.values, dtype=torch.float64)
labels = torch.tensor(labels.values, dtype=torch.long) - 1 # subtract 1 to start at 0

In [ ]:
def init():
    df = pd.read_csv('https://gist.githubusercontent.com/tijptjik/9408623/raw/b237fa5848349a14a14e5d4107dc7897c21951f5/wine.csv')

    features = df.drop(columns = 'Wine')
    labels = df['Wine']

    features = torch.tensor(features.values, dtype=torch.float64)
    labels = torch.tensor(labels.values, dtype=torch.long) - 1 # subtract 1 to start at 0

    return features, labels

In [ ]:
features, labels = init()
features, labels

(tensor([[1.4230e+01, 1.7100e+00, 2.4300e+00,  ..., 1.0400e+00, 3.9200e+00,
          1.0650e+03],
         [1.3200e+01, 1.7800e+00, 2.1400e+00,  ..., 1.0500e+00, 3.4000e+00,
          1.0500e+03],
         [1.3160e+01, 2.3600e+00, 2.6700e+00,  ..., 1.0300e+00, 3.1700e+00,
          1.1850e+03],
         ...,
         [1.3270e+01, 4.2800e+00, 2.2600e+00,  ..., 5.9000e-01, 1.5600e+00,
          8.3500e+02],
         [1.3170e+01, 2.5900e+00, 2.3700e+00,  ..., 6.0000e-01, 1.6200e+00,
          8.4000e+02],
         [1.4130e+01, 4.1000e+00, 2.7400e+00,  ..., 6.1000e-01, 1.6000e+00,
          5.6000e+02]], dtype=torch.float64),
 tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

## Section 2: The ``__len__`` and the `__getitem__` functions and making the custom data class

In this notebook, you'll learn how to create each of this methods and make a custom data class. You'll start with what goes into the ``__init__`` function.

| Code | Description |
|---|---|
| `len(df)` | Returns the the number of rows in the DataFrame ``df``. |
| `x[0]` | Get the first element of `x`. |
| `x[-1]` | Get the last element of `x`. |
| `x[25]` | Get the element of `x` at index 25. |

**TODO**: Add material on classes and how to use ``self``.

#### **Exercises**

**Exercise**: Get the number of passengers in the Titanic DataFrame.

**Hint**: It will correspond to the number of rows in the DataFrame.

In [43]:
len(df_titanic)

891

**Exercise**: Get the number of wines in the wine DataFrame.

In [44]:
len(df)

178

**Exercise**: Make a function named `get_length_df` that takes a DataFrame as an argument and returns the number of rows in the DataFrame.

In [ ]:
def get_length_df(df):
    return len(df)

**Exercise**: Get the data at index 0 in the `features` variable.

In [53]:
features[0]

tensor([1.4230e+01, 1.7100e+00, 2.4300e+00, 1.5600e+01, 1.2700e+02, 2.8000e+00,
        3.0600e+00, 2.8000e-01, 2.2900e+00, 5.6400e+00, 1.0400e+00, 3.9200e+00,
        1.0650e+03], dtype=torch.float64)

**Exercise**: Get the data at index 0 in the `labels` variable.

In [54]:
labels[0]

tensor(0)

**Exercise**: Get the data at the last index in the `labels` variable.

In [57]:
labels[-1]

tensor(2)

**Exercise**: Complete the function `getitem` below, which takes a variable ``data`` and an ``index`` as arguments and returns the item at that index in the data.

In [58]:
def getitem(data):
    item = ___
    return item

**Example**: Make a custom class named `TitanicDataset`. The ``__init__`` function of the class should load the titanic data, extract the data in the columns `age` and `fare` to be used as features, and extract the data in the `survived` column to be used as labels. The `__len__` and `__getitem__` functions should return the number of samples (passengers) in the data and the data at a given ``index``, respectively.

In [60]:
class TitanicDataset(Dataset):
    def __init__(self):
        '''Load dataset and extract features and labels'''
        super().__init__()

        # load the data
        df_titanic = pd.read_csv('data/raw/titanic.csv')

        # extract features and labels and turn them into tensors
        self.features = torch.tensor(df_titanic[['age', 'fare']].values, dtype = torch.float32)
        self.labels = torch.tensor(df_titanic['survived'].values, dtype=torch.long)

    def __len__(self):
        '''Get the number of samples in the dataset'''
        return len(self.labels)
    
    def __getitem__(self, index):
        '''Get the item and a given index in the dataset'''
        return self.features[index], self.labels[index]


In [62]:
titanic = TitanicDataset()
titanic

**Excercise**: Complete the code below to make a custom class named `WineDataset`. The ``__init__`` function of the class should load the wine data and extract the same data to be used as features and labels as in section 1 above. The `__len__` and `__getitem__` functions should return the number of samples (wines) in the data and the data at a given ``index``, respectively.

In [ ]:
class WineDataset(Dataset):
    def __init__():
        '''Load dataset and extract features and labels'''
        super.__init__()

        # load the wine data into a pandas dataframe

        # extract the relevant columns and initialize a features and a labels tensor

    def __len__(self):
        '''Get the number of samples in the dataset'''
        # enter code here

    def __getitem__(self, index):
        '''Get the item and a given index in the dataset'''
        # enter code here

In [ ]:
class WineDataset(Dataset):
    def __init__(self):
        # load data
        df = pd.read_csv('data/raw/wine.csv')

        self.features = torch.tensor(df.drop(columns = 'Wine').values, dtype=torch.float32)
        self.labels = torch.tensor(df['Wine'].values, dtype = torch.long) - 1 # subtract 1 to start at 0

    def __len__(self):
        '''Get the number of samples in the dataset'''
        return len(self.labels)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [128]:
wine = WineDataset()
wine

## Section 3: The DataLoader

| Code | Description |
|---|---|
| `dataloader = DataLoader(DatasetName())` | Creates a DataLoader for the custom dataset class `DatasetName` and assigns it to the variable `dataloader`. |
| ` DataLoader(DatasetName(), shuffle=True, batch_size=some_number)` | Creates a DataLoader, shuffles the samples in the data, and sets the batch size to `some_number` (default value is 1). |
| `iter(iterable_object)` | `iter` makes an iterator out of an `iterable object`. |
| `next(iter(iterable_object))` | `next` returns the next item in the iterable object. |
| `for i in range(5):` <br> &nbsp;&nbsp;&nbsp;&nbsp; `print(i)` | Print every number `i` between 0 and 5 (not including 5)|

#### **Exercises**


**Example**: Create a dataloader for the Titanic dataset. Get the features and the label for a single passenger in the dataset.

**Hint**: To get the data on a single row in the dataset, you can use the built-in Python functions `next` and `iter`. This way, you can get a "peek" at the data in the dataloader.

In [129]:
titanic_dataloader = DataLoader(TitanicDataset())

feature, label = next(iter(titanic_dataloader))
feature, label

(tensor([[22.0000,  7.2500]]), tensor([0]))

**Exercise**: Create a dataloader named `wine_dataloader` for the Wine dataset.

In [153]:
wine_dataloader = DataLoader(WineDataset())

**Exercise**: Get the features and the wine label for a single wine in the dataset.

In [155]:
feature, label = next(iter(wine_dataloader))
feature, label

(tensor([[1.4230e+01, 1.7100e+00, 2.4300e+00, 1.5600e+01, 1.2700e+02, 2.8000e+00,
          3.0600e+00, 2.8000e-01, 2.2900e+00, 5.6400e+00, 1.0400e+00, 3.9200e+00,
          1.0650e+03]]),
 tensor([0]))

**Exercise**: Run the cell with the solution to the previous exercise again. Did you get the same feature values or different ones?

In [132]:
# same

You should have gotten the same feature values and label in the previous exercise no matter how many times you reran the code cell. If you want to draw a random set of feature values and label from the daataset, you need to set `shuffle = True` when you create the dataloader.

**Exercise**: Create a dataloader for the wine dataset again, but this time, set the parameter `shuffle = True` in the ``DataLoader`` constructor. 

In [133]:
wine_dataloader = DataLoader(WineDataset(), shuffle=True)

**Exercise**: Get the features and the wine label for a single wine in the dataset. Do you get different numbers each time your run it now? Why do you/don't you, do you think?

In [134]:
feature, label = next(iter(wine_dataloader))
feature, label

(tensor([[1.1450e+01, 2.4000e+00, 2.4200e+00, 2.0000e+01, 9.6000e+01, 2.9000e+00,
          2.7900e+00, 3.2000e-01, 1.8300e+00, 3.2500e+00, 8.0000e-01, 3.3900e+00,
          6.2500e+02]]),
 tensor([1]))

You can also set the batch size - how many samples of your dataset you batch together - in the ``DataLoader``. By default, the batch size is 1.

**Exercise**: Create the wine dataloader again with `shuffle = True` and set `batch_size = 8`. Get the features and the wine labels for a *batch* of wines in the dataset.

In [135]:
wine_dataloader = DataLoader(WineDataset(), shuffle=True, batch_size=8)

In [136]:
feature, label = next(iter(wine_dataloader))
feature, label

(tensor([[1.2510e+01, 1.7300e+00, 1.9800e+00, 2.0500e+01, 8.5000e+01, 2.2000e+00,
          1.9200e+00, 3.2000e-01, 1.4800e+00, 2.9400e+00, 1.0400e+00, 3.5700e+00,
          6.7200e+02],
         [1.3750e+01, 1.7300e+00, 2.4100e+00, 1.6000e+01, 8.9000e+01, 2.6000e+00,
          2.7600e+00, 2.9000e-01, 1.8100e+00, 5.6000e+00, 1.1500e+00, 2.9000e+00,
          1.3200e+03],
         [1.4750e+01, 1.7300e+00, 2.3900e+00, 1.1400e+01, 9.1000e+01, 3.1000e+00,
          3.6900e+00, 4.3000e-01, 2.8100e+00, 5.4000e+00, 1.2500e+00, 2.7300e+00,
          1.1500e+03],
         [1.3690e+01, 3.2600e+00, 2.5400e+00, 2.0000e+01, 1.0700e+02, 1.8300e+00,
          5.6000e-01, 5.0000e-01, 8.0000e-01, 5.8800e+00, 9.6000e-01, 1.8200e+00,
          6.8000e+02],
         [1.2000e+01, 9.2000e-01, 2.0000e+00, 1.9000e+01, 8.6000e+01, 2.4200e+00,
          2.2600e+00, 3.0000e-01, 1.4300e+00, 2.5000e+00, 1.3800e+00, 3.1200e+00,
          2.7800e+02],
         [1.2080e+01, 1.1300e+00, 2.5100e+00, 2.4000e+01, 7.8000e

**Example**: Loop through all batches in the titanic dataloader created below and do something ...

In [137]:
titanic_dataloader = DataLoader(TitanicDataset(), shuffle=True, batch_size=8, )

In [138]:
for idx_batch, (features_batch, labels_batch) in enumerate(titanic_dataloader):
    print(f'Batch nr. {idx_batch}; labels this batch: {labels_batch}')

Batch nr. 0; labels this batch: tensor([1, 0, 1, 0, 0, 1, 0, 0])
Batch nr. 1; labels this batch: tensor([0, 0, 0, 1, 0, 0, 0, 0])
Batch nr. 2; labels this batch: tensor([1, 1, 0, 0, 1, 0, 0, 0])
Batch nr. 3; labels this batch: tensor([1, 1, 1, 1, 0, 0, 0, 0])
Batch nr. 4; labels this batch: tensor([0, 0, 1, 0, 0, 1, 0, 0])
Batch nr. 5; labels this batch: tensor([1, 0, 0, 0, 0, 1, 1, 0])
Batch nr. 6; labels this batch: tensor([1, 1, 1, 0, 0, 1, 1, 1])
Batch nr. 7; labels this batch: tensor([0, 0, 1, 0, 1, 0, 0, 1])
Batch nr. 8; labels this batch: tensor([1, 0, 0, 0, 0, 0, 0, 0])
Batch nr. 9; labels this batch: tensor([0, 0, 0, 0, 1, 1, 1, 1])
Batch nr. 10; labels this batch: tensor([0, 0, 0, 0, 1, 0, 0, 0])
Batch nr. 11; labels this batch: tensor([0, 1, 0, 0, 0, 0, 1, 0])
Batch nr. 12; labels this batch: tensor([1, 0, 0, 0, 0, 0, 0, 0])
Batch nr. 13; labels this batch: tensor([0, 0, 0, 0, 1, 1, 1, 1])
Batch nr. 14; labels this batch: tensor([0, 0, 0, 1, 0, 0, 1, 0])
Batch nr. 15; labels

**Exercise**: Loop through all batches in the wine dataloader created below and do something ...

In [159]:
wine_dataloader = DataLoader(WineDataset(), shuffle=True, batch_size=8)

In [160]:
for idx_batch, (features_batch, labels_batch) in enumerate(wine_dataloader):
        print(f'Batch nr. {idx_batch}; labels this batch: {labels_batch}')

Batch nr. 0; labels this batch: tensor([0, 0, 0, 0, 2, 0, 2, 2])
Batch nr. 1; labels this batch: tensor([1, 0, 1, 0, 0, 1, 1, 0])
Batch nr. 2; labels this batch: tensor([1, 2, 2, 2, 2, 0, 1, 0])
Batch nr. 3; labels this batch: tensor([0, 2, 0, 2, 0, 0, 0, 2])
Batch nr. 4; labels this batch: tensor([2, 0, 2, 1, 2, 1, 2, 2])
Batch nr. 5; labels this batch: tensor([2, 0, 0, 1, 2, 2, 1, 0])
Batch nr. 6; labels this batch: tensor([1, 1, 0, 2, 0, 0, 2, 2])
Batch nr. 7; labels this batch: tensor([0, 1, 0, 1, 1, 2, 0, 1])
Batch nr. 8; labels this batch: tensor([2, 2, 1, 1, 1, 1, 0, 0])
Batch nr. 9; labels this batch: tensor([2, 1, 1, 1, 2, 0, 1, 0])
Batch nr. 10; labels this batch: tensor([1, 0, 2, 1, 2, 2, 1, 1])
Batch nr. 11; labels this batch: tensor([0, 1, 1, 2, 0, 1, 0, 0])
Batch nr. 12; labels this batch: tensor([2, 0, 0, 0, 1, 1, 1, 1])
Batch nr. 13; labels this batch: tensor([1, 0, 2, 1, 0, 1, 1, 2])
Batch nr. 14; labels this batch: tensor([1, 2, 1, 1, 1, 2, 0, 2])
Batch nr. 15; labels

## Section 4: Split data into train, validation, and test data.

| Code | Description |
|---|---|
| `subdatasetA, subdatasetB, ... = random_split(NameDataset(), lengths = [fractionA, fractionB, ...])` | Splits the data in `NameDataset` into subdatasets. The fractions in `lengths` specifies the proportion of the data that should go into each subset. |
| `subdatasetA, subdatasetB, ... = random_split(NameDataset(), lengths = [fractionA, fractionB, ...], generator=some_torch_generator)` | The parameter `generator` can be fixed to make sure that the data is split in exactly the same way each time. This ensures reproducibility. |
| `torch.Generator().manual_seed(some_num)` | Sets the seed. |

#### **Exercises**

**Example**: Split the dataset into three subsets according to the fractions provided below: a dataset used for training the model, a dataset used for validation during training, and a dataset used to test the final model.

In [ ]:
train_fraction = 0.7
val_fraction = 0.15
test_fraction = 1-train_fraction-val_fraction

train_data, val_data, test_data = random_split(WineDataset(), lengths=[train_fraction, val_fraction, test_fraction])

**Exercise**: Set the fraction of the data going into the train dataset to 0.8 and the fractions going into the validation and test subsets to 0.1. Then split the data again.

In [ ]:
train_fraction = 0.8
val_fraction = 0.1
test_fraction = 1-train_fraction-val_fraction

train_data, val_data, test_data = random_split(WineDataset(), lengths=[train_fraction, val_fraction, test_fraction])

**Exercise**: Split the dataset into only two datasets: a train and a test subset. Set the fraction of the data going into the train dataset to 0.8 and the fraction going into test subset to 0.2. Then split the data again.

In [268]:
train_fraction = 0.8
test_fraction = 0.2

train_data, test_data = random_split(WineDataset(), lengths=[train_fraction, test_fraction])

**Exercise**: Create a torch random number generator from the seed below.

In [ ]:
SEED = 42

In [269]:
SEED = 42
generator = torch.Generator().manual_seed(SEED)
generator

**Exercise**: Create a torch random generator from the seed below. Then split the data with the same fractions as in the example at the beginning of the section, but this time, pass the generator as a parameter to the `random_split` function.

In [288]:
SEED = 42
generator = torch.Generator().manual_seed(SEED)

train_fraction = 0.7
val_fraction = 0.15
test_fraction = 1-train_fraction-val_fraction

train_data, val_data, test_data = random_split(WineDataset(), lengths=[train_fraction, val_fraction, test_fraction], generator=generator)

**Exercise**: Copy the code in the cell below and add it at the bottom of your solution in the previous exercise so the features and a label fetched from the training subset is displayed after splitting. Do you get the same or different values when you run the code repeatedly?

In [289]:
next(iter(train_data))

(tensor([1.2080e+01, 2.0800e+00, 1.7000e+00, 1.7500e+01, 9.7000e+01, 2.2300e+00,
         2.1700e+00, 2.6000e-01, 1.4000e+00, 3.3000e+00, 1.2700e+00, 2.9600e+00,
         7.1000e+02]),
 tensor(1))

**Exercise**: Make a dataloader out of the ``train_data`` subset. Set the batch size to a number of your choice and pass a random generator to the dataloader.

In [ ]:
train_dataloader = DataLoader(train_data, shuffle=True, batch_size=8, generator=generator)


**Exercise**: Make a dataloader out of the ``val_data`` and `test_data` subsets as wel. Set the batch size to a number of your choice and pass a random generator to the dataloader.

In [291]:
val_dataloader = DataLoader(val_data, shuffle=True, batch_size=len(val_data), generator=generator)
test_dataloader = DataLoader(test_data, shuffle=True, batch_size=len(test_data), generator=generator)